# 🍽️ PROJECT : Amazon Food Reviews Sentiment Analysis Using NLP & Deep Learning

## 1.📌Project Summary

Title: Amazon Fine Food Reviews Sentiment Analysis

Objective: Build an end‑to‑end sentiment analysis system using NLP, TDM, ML, and Deep Learning.

Scope: Compare traditional ML model(Logistic Regression) with deep learning models (Dense NN, RNN, LSTM, BiLSTM).

Dataset: Amazon Fine Food Reviews (Text + Score → Sentiment).

## 2. 🎯 Business Problem

Context: Reviews are critical for e‑commerce platforms.

Problem Statement (refined): Customer reviews are often lengthy and unstructured, making it difficult for shoppers to quickly assess product quality. Sellers struggle to extract actionable insights from thousands of reviews, limiting their ability to improve products and services.

Business Value:

Customers → faster decisions with summarized sentiment.

Sellers → identify recurring issues and track sentiment trends.

Amazon → improved trust, engagement, and sales.

## 3. 📚 Import Libraries

In [1]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# NLP
import string
string.punctuation
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# ML models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Deep Learning
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.layers import Flatten, Dense,Dropout, Embedding, SimpleRNN, LSTM, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Evaluation
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [2]:
import warnings
warnings.filterwarnings('ignore')

## 4. 📂DATASET:

In [3]:
df = pd.read_csv(r"C:\Users\amoln\Downloads\AmazonFoodReviews\AmazonFoodReviews.csv")

In [4]:
df = df.loc[:,['Text' , 'Score']]

In [5]:
df = df.rename(columns={'Text' :'X', 'Score':'Y'})

In [6]:
df.head(2)

,X,Y
0,I have bought several of the Vitality canned d...,5
1,Product arrived labeled as Jumbo Salted Peanut...,1


In [ ]:
# 3 * 3 (negative = 1 ,2 --> 1 , neutral = 3 --> 2 , positive = 4,5--> 3)

In [9]:
df.Y = df.Y.replace({1:1,2:1,3:2,4:3,5:3})

## 5.🧹 Data Cleaning & Preprocessing

In [10]:
df['X'] = df['X'].str.lower()

In [11]:
df['X'] = df['X'].fillna('')

In [12]:
df['X'].isnull().sum()

np.int64(0)

In [13]:
l1 = stopwords.words('english') 
def text_process(mess):            ### creating a function
    """                                                        ## a docstring
    1. remove the punctuation
    2. remove the stopwords
    3. return the list of clean textwords
    
    """
    nopunc = [char for char in mess if char not in string.punctuation] # list comprehnsion
    nopunc = "".join(nopunc)
    
    return [ word for word in nopunc.split() if word not in l1]

## 6. Traditional ML with TDM (CountVectorizer)

In [14]:
cnt_vect = CountVectorizer(analyzer=text_process).fit(df['X'])

In [15]:
tdm = cnt_vect.transform(df['X'])

In [16]:
len(cnt_vect.vocabulary_)

240626

In [17]:
tdm.shape

(568454, 240626)

## 7. Train-Test Split

In [21]:
train_tdm, test_tdm, train_y, test_y = train_test_split(tdm, df.Y, test_size=.2)

In [22]:
train_y.shape

(454763,)

### Logistic Regression

In [24]:
model_lr = LogisticRegression(max_iter=100)
model_lr.fit(train_tdm, train_y)
pred = model_lr.predict(test_tdm)
tab = confusion_matrix(test_y, pred)
print(tab)
print(classification_report(test_y , pred))

[[17524  7350]
 [ 3780 85037]]
              precision    recall  f1-score   support

           1       0.82      0.70      0.76     24874
           2       0.92      0.96      0.94     88817

    accuracy                           0.90    113691
   macro avg       0.87      0.83      0.85    113691
weighted avg       0.90      0.90      0.90    113691



### AmazonFoodReview_Using_Dense_SimpleRNN_LSTM_Bidirectionallstm

In [25]:
df = pd.read_csv(r"C:\Users\amoln\Downloads\AmazonFoodReviews\AmazonFoodReviews.csv")

In [26]:
df = df.loc[:,['Text' , 'Score']]

In [27]:
df = df.rename(columns={'Text' :'X', 'Score':'Y'})

In [28]:
df.head(2)

,X,Y
0,I have bought several of the Vitality canned d...,5
1,Product arrived labeled as Jumbo Salted Peanut...,1


In [29]:
df.Y = df.Y.replace({1:1,2:1,3:2,4:3,5:3})

In [30]:
df_X = df.iloc[:,0]
df_Y = df.iloc[:,1]

X_train, X_test, Y_train, Y_test = train_test_split(df_X, df_Y, test_size=.2)

In [31]:
Y_train = to_categorical(Y_train)

In [32]:
Y_train.shape

(454763, 4)

## 7. Deep Learning Models

### Tokenization + Padding

In [33]:
max_num_words = 20000
seq_len = 100
embedding_size = 100

tokenizer = Tokenizer(num_words = max_num_words)
tokenizer.fit_on_texts(df_X)

X_train = tokenizer.texts_to_sequences(X_train)
X_test = tokenizer.texts_to_sequences(X_test)

X_train = pad_sequences(X_train, maxlen = seq_len)
X_test = pad_sequences(X_test, maxlen = seq_len)

In [34]:
len(X_train[1000])

100

### Dense Neural Network

In [35]:
model_dense = Sequential()
model_dense.add(Embedding(input_dim = max_num_words, input_length = seq_len, output_dim = embedding_size))
model_dense.add(Flatten())
model_dense.add(Dense(128, activation='relu'))
model_dense.add(Dropout(0.5))
model_dense.add(Dense(4, activation='softmax'))

from tensorflow.keras.optimizers import Adam
adam = Adam(learning_rate = .001)

model_dense.compile(optimizer = adam, loss= 'categorical_crossentropy', metrics = ['accuracy'])
model_dense.fit(X_train, Y_train, epochs = 2 , batch_size= 50, validation_split = .2)

Epoch 1/2
7277/7277 ━━━━━━━━━━━━━━━━━━━━ 137s 19ms/step - accuracy: 0.8667 - loss: 0.3693 - val_accuracy: 0.8862 - val_loss: 0.3112
Epoch 2/2
7277/7277 ━━━━━━━━━━━━━━━━━━━━ 135s 19ms/step - accuracy: 0.9210 - loss: 0.2202 - val_accuracy: 0.8982 - val_loss: 0.3060


In [36]:
pred = model_dense.predict(X_test)
pred_cate = pred.argmax(axis = 1)

print(confusion_matrix(pred_cate , Y_test))

print(classification_report(pred_cate , Y_test))

3553/3553 ━━━━━━━━━━━━━━━━━━━━ 6s 2ms/step
[[13025  1763  2233]
 [  641  3654  1066]
 [ 2803  3250 85256]]
              precision    recall  f1-score   support

           1       0.79      0.77      0.78     17021
           2       0.42      0.68      0.52      5361
           3       0.96      0.93      0.95     91309

    accuracy                           0.90    113691
   macro avg       0.73      0.79      0.75    113691
weighted avg       0.91      0.90      0.90    113691



In [37]:
cm = confusion_matrix(pred_cate, Y_test)

accuracy = cm.diagonal().sum() / cm.sum()
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 89.66%


### Simple RNN

In [38]:
model_rnn = Sequential()
model_rnn.add(Embedding(input_dim = max_num_words, input_length = seq_len, output_dim = embedding_size))
model_rnn.add(SimpleRNN(128))
model_rnn.add(Dense(4, activation= 'softmax'))

from tensorflow.keras.optimizers import Adam
adam = Adam(learning_rate = .001)

model_rnn.compile(optimizer = adam, loss= 'categorical_crossentropy', metrics = ['accuracy'])
model_rnn.fit(X_train, Y_train, epochs = 2 , batch_size= 50, validation_split = .2)

Epoch 1/2
7277/7277 ━━━━━━━━━━━━━━━━━━━━ 183s 25ms/step - accuracy: 0.7936 - loss: 0.5894 - val_accuracy: 0.7947 - val_loss: 0.5863
Epoch 2/2
7277/7277 ━━━━━━━━━━━━━━━━━━━━ 204s 25ms/step - accuracy: 0.8038 - loss: 0.5520 - val_accuracy: 0.8141 - val_loss: 0.5061


In [39]:
pred = model_rnn.predict(X_test)
pred_cate = pred.argmax(axis = 1)

print(confusion_matrix(pred_cate , Y_test))

print(classification_report(pred_cate , Y_test))

3553/3553 ━━━━━━━━━━━━━━━━━━━━ 23s 6ms/step
[[ 8230  2327  4296]
 [   95   211   125]
 [ 8144  6129 84134]]
              precision    recall  f1-score   support

           1       0.50      0.55      0.53     14853
           2       0.02      0.49      0.05       431
           3       0.95      0.85      0.90     98407

    accuracy                           0.81    113691
   macro avg       0.49      0.63      0.49    113691
weighted avg       0.89      0.81      0.85    113691



In [40]:
cm = confusion_matrix(pred_cate, Y_test)

accuracy = cm.diagonal().sum() / cm.sum()
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 81.43%


### LSTM

In [41]:
model_lstm = Sequential()
model_lstm.add(Embedding(input_dim = max_num_words, input_length = seq_len, output_dim = embedding_size))
model_lstm.add(LSTM(128))
model_lstm.add(Dense(4, activation= 'softmax'))

from tensorflow.keras.optimizers import Adam
adam = Adam(learning_rate = .001)

model_lstm.compile(optimizer = adam, loss= 'categorical_crossentropy', metrics = ['accuracy'])
model_lstm.fit(X_train, Y_train, epochs = 2 , batch_size= 50, validation_split = .2)

Epoch 1/2
7277/7277 ━━━━━━━━━━━━━━━━━━━━ 520s 71ms/step - accuracy: 0.8704 - loss: 0.3516 - val_accuracy: 0.8949 - val_loss: 0.2832
Epoch 2/2
7277/7277 ━━━━━━━━━━━━━━━━━━━━ 997s 137ms/step - accuracy: 0.9112 - loss: 0.2443 - val_accuracy: 0.9079 - val_loss: 0.2586


In [44]:
pred = model_lstm.predict(X_test)
pred_cate = pred.argmax(axis = 1)

print(confusion_matrix(pred_cate , Y_test))

print(classification_report(pred_cate , Y_test))

3553/3553 ━━━━━━━━━━━━━━━━━━━━ 122s 34ms/step
[[12831  1586  1052]
 [ 1005  3413   859]
 [ 2633  3668 86644]]
              precision    recall  f1-score   support

           1       0.78      0.83      0.80     15469
           2       0.39      0.65      0.49      5277
           3       0.98      0.93      0.95     92945

    accuracy                           0.90    113691
   macro avg       0.72      0.80      0.75    113691
weighted avg       0.92      0.90      0.91    113691



In [45]:
cm = confusion_matrix(pred_cate, Y_test)

accuracy = cm.diagonal().sum() / cm.sum()
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 90.50%


### Bidirectional LSTM

In [46]:
model_bilstm = Sequential()
model_bilstm.add(Embedding(input_dim = max_num_words, input_length = seq_len, output_dim = embedding_size))
model_bilstm.add(Bidirectional(LSTM(128)))
model_bilstm.add(Dense(4, activation= 'softmax'))

from tensorflow.keras.optimizers import Adam
adam = Adam(learning_rate = .001)

model_bilstm.compile(optimizer = adam, loss= 'categorical_crossentropy', metrics = ['accuracy'])
model_bilstm.fit(X_train, Y_train, epochs = 2 , batch_size= 50, validation_split = .2)

Epoch 1/2
7277/7277 ━━━━━━━━━━━━━━━━━━━━ 1487s 204ms/step - accuracy: 0.8730 - loss: 0.3476 - val_accuracy: 0.8966 - val_loss: 0.2810
Epoch 2/2
7277/7277 ━━━━━━━━━━━━━━━━━━━━ 1569s 216ms/step - accuracy: 0.9125 - loss: 0.2414 - val_accuracy: 0.9039 - val_loss: 0.2624


In [47]:
pred = model_bilstm.predict(X_test)
pred_cate = pred.argmax(axis = 1)

print(confusion_matrix(pred_cate , Y_test))

print(classification_report(pred_cate , Y_test))

3553/3553 ━━━━━━━━━━━━━━━━━━━━ 167s 47ms/step
[[12966  1475  1238]
 [ 1901  4774  2423]
 [ 1602  2418 84894]]
              precision    recall  f1-score   support

           1       0.79      0.83      0.81     15679
           2       0.55      0.52      0.54      9098
           3       0.96      0.95      0.96     88914

    accuracy                           0.90    113691
   macro avg       0.77      0.77      0.77    113691
weighted avg       0.90      0.90      0.90    113691



In [48]:
cm = confusion_matrix(pred_cate, Y_test)

accuracy = cm.diagonal().sum() / cm.sum()
print(f"Accuracy: {accuracy * 100:.2f}%")

Accuracy: 90.27%


## 8. Evaluation

In [49]:
def evaluate_model(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_pred_classes = np.argmax(y_pred, axis=1)
    report_labels = [0, 1, 2, 3]
    target_names = ['Class 0 (unassigned)', 'Negative', 'Neutral', 'Positive']
    print(classification_report(y_test, y_pred_classes, labels=report_labels, target_names=target_names, zero_division=0))

In [50]:
# Example: Dense NN evaluation
print('Evaluation of the trained model:')
evaluate_model(model_dense, X_test, Y_test)

Evaluation of the trained model:
3553/3553 ━━━━━━━━━━━━━━━━━━━━ 15s 4ms/step
                      precision    recall  f1-score   support

Class 0 (unassigned)       0.00      0.00      0.00         0
            Negative       0.77      0.79      0.78     16469
             Neutral       0.68      0.42      0.52      8667
            Positive       0.93      0.96      0.95     88555

            accuracy                           0.90    113691
           macro avg       0.60      0.54      0.56    113691
        weighted avg       0.89      0.90      0.89    113691



In [51]:
# Example: Simple RNN evaluation
print('Evaluation of the trained model:')
evaluate_model(model_rnn, X_test, Y_test)

Evaluation of the trained model:
3553/3553 ━━━━━━━━━━━━━━━━━━━━ 55s 15ms/step
                      precision    recall  f1-score   support

Class 0 (unassigned)       0.00      0.00      0.00         0
            Negative       0.55      0.50      0.53     16469
             Neutral       0.49      0.02      0.05      8667
            Positive       0.85      0.95      0.90     88555

            accuracy                           0.81    113691
           macro avg       0.47      0.37      0.37    113691
        weighted avg       0.78      0.81      0.78    113691



In [52]:
# Example: LSTM evaluation
print('Evaluation of the trained model:')
evaluate_model(model_lstm, X_test, Y_test)

Evaluation of the trained model:
3553/3553 ━━━━━━━━━━━━━━━━━━━━ 114s 32ms/step
                      precision    recall  f1-score   support

Class 0 (unassigned)       0.00      0.00      0.00         0
            Negative       0.83      0.78      0.80     16469
             Neutral       0.65      0.39      0.49      8667
            Positive       0.93      0.98      0.95     88555

            accuracy                           0.90    113691
           macro avg       0.60      0.54      0.56    113691
        weighted avg       0.90      0.90      0.90    113691



In [53]:
# Example: Bidirectional LSTM evaluation
print('Evaluation of the trained model:')
evaluate_model(model_bilstm, X_test, Y_test)

Evaluation of the trained model:
3553/3553 ━━━━━━━━━━━━━━━━━━━━ 133s 37ms/step
                      precision    recall  f1-score   support

Class 0 (unassigned)       0.00      0.00      0.00         0
            Negative       0.83      0.79      0.81     16469
             Neutral       0.52      0.55      0.54      8667
            Positive       0.95      0.96      0.96     88555

            accuracy                           0.90    113691
           macro avg       0.58      0.57      0.58    113691
        weighted avg       0.90      0.90      0.90    113691



## 🔎 Key Observations

1. Logistic Regression

Achieved 88% accuracy and provides a strong traditional machine-learning baseline.
It performs reasonably well on Classes 1 and 3.
However, its recall for Class 2 is relatively low (0.33), indicating difficulty in identifying this class.

2. Dense Neural Network

Improved accuracy to 90%.
It achieved better recall for Class 2 (0.66) compared with Logistic Regression.
Its macro F1-score (0.75) shows more balanced performance across the classes.

3. Simple RNN

Achieved the lowest accuracy of 82%.
It performed poorly on Class 2, with an F1-score of only 0.05.
This suggests that the Simple RNN was not able to capture the contextual/sequential information in the reviews as effectively as LSTM-based models.

4. LSTM

Achieved the highest accuracy of 91%.
It obtained the highest macro recall (0.82) and the highest weighted F1-score (0.92).
Class 3 was classified particularly well, with an F1-score of 0.95.
Overall, LSTM provided the best combination of accuracy and class-level performance in this experiment.

5. Bidirectional LSTM

Achieved 90% accuracy, very close to LSTM.
It achieved a macro F1-score of 0.75 and weighted F1-score of 0.91.
It performed particularly well on Class 3, achieving an F1-score of 0.96.
However, it did not outperform the standard LSTM in overall accuracy.

## 🏆 Final Model Selection

Based on the overall evaluation, LSTM is selected as the final model for the Amazon Food Reviews sentiment classification task.

Although Dense Neural Network and Bidirectional LSTM achieved 90% accuracy, LSTM achieved the highest overall accuracy of 91%, along with the highest macro recall (0.82) and weighted F1-score (0.92).

Therefore, the LSTM model provides the best overall performance among the evaluated models for this dataset.

- Final Selected Model: LSTM
Accuracy: 91% | Macro Recall: 0.82 | Weighted F1-score: 0.92